### Make .csv of CN coordinates

In [ ]:
import os

folder = r'C:\Users\andre\Desktop\CN-Parfile\CN-Parfile'
coordList_output = r'C:\Users\andre\Desktop\CN-coords.csv'

files = os.listdir(folder)

with open(coordList_output, 'w') as fo:
  fo.write('stationID,x,y\n')
  for f in files:
    with open(os.path.join(folder, f)) as fi:
      lines = fi.readlines()  
    coord_line = lines[84]
    x = coord_line.split(',')[0].split()[1]
    y = coord_line.split(',')[1].split()[1]
    fo.write(','.join([f[:-4], x, y]) + '\n')



### Make Allstations.par

In [5]:
import os
folder = r'C:\Users\andre\Desktop\parfiles-2015'
outfile = r'C:\Users\andre\Desktop\Allstations.par'

files = os.listdir(folder)

with open(outfile, 'w') as fo:
  for f in files:
    with open(os.path.join(folder, f)) as fi:  
      l = fi.read()
    fo.write(l)


### Run FindMatch for all par files

In [1]:
from subprocess import Popen, PIPE
import shutil
import os

parFolder = r'C:\Users\andre\Desktop\CN-Parfile\CN-Parfile'
outputFolder = r'C:\Users\andre\Desktop\output'
intlFolder = r'C:\Users\andre\Desktop\International\International'

parFiles = os.listdir(parFolder)
topFiles = [file.strip('.par') + '.top' for file in parFiles]

def make_tops(parFile, topFile, parDir, intlDir):
  with open(os.path.join(parDir, parFile)) as f_in:
    lines = f_in.readlines()
  with open(os.path.join(intlDir, topFile), 'w') as f_out:
    for line in lines[:12]:
      f_out.write(line)
  return 0

def run_intl(topFile):
  p = Popen(['FindMatch'], cwd=intlFolder, stdin=PIPE, stdout=PIPE, shell=True)
  p.communicate(bytes(topFile + '\r\n\r\n', 'utf-8'))
  return 0

def clear_files(parFile, topFile, parDir, intlDir, outputDir):
  os.remove(os.path.join(intlDir, topFile))        
  shutil.copyfile(os.path.join(intlDir, parFile), os.path.join(outputDir, parFile))
  os.remove(os.path.join(intlDir, parFile))        
  return 0

for i, file in enumerate(parFiles):
  output = make_tops(parFiles[i], topFiles[i], parFolder, intlFolder)
  output = run_intl(topFiles[i])
  output = clear_files(parFiles[i], topFiles[i], parFolder, intlFolder, outputFolder)


### Javascript for GEE download of Dewpoint Temperature

In [ ]:
var points_fc = ee.FeatureCollection('projects/ee-andrewfullhart/assets/CN-coords');

var area_shp = ee.Geometry.BBox(68.0, 15.5, 137.5, 54.0);
Map.addLayer(area_shp);
var era_ic = ee.ImageCollection('ECMWF/ERA5/MONTHLY');

var first_im = era_ic.first();
var scale = first_im.projection().nominalScale().getInfo();

var order_months = ee.List([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]);
var string_months = ee.List(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']);

var start_year = ee.Number(1986);
var end_year = ee.Number(2015);
var start = ee.Date.fromYMD(ee.Number(start_year), 1, 1);
var end = ee.Date.fromYMD(ee.Number(end_year).add(1), 1, 1);
var year_ic = era_ic.filterDate(start, end).select('dewpoint_2m_temperature');

function month_fn(mobj){
  var m = ee.Number(mobj);
  var month_ic = year_ic.filter(ee.Filter.calendarRange(m, m, 'month'));
  var mean_im = month_ic.reduce(ee.Reducer.mean()).clip(area_shp).add(-273.15);
  return mean_im;
}

var mean_ic_list = order_months.map(month_fn);

Map.addLayer(points_fc);

function point_fn(ftobj){

  var ft = ee.Feature(ftobj);
  var id = ft.get('stationID');
  
  function sample_fn(mobj){
    var m = ee.Number(mobj);
    var sample_fc = ee.Image(mean_ic_list.get(m.subtract(1))).sampleRegions({collection:ee.FeatureCollection([ft]), scale:scale, tileScale:4});
    var value = sample_fc.first().get('dewpoint_2m_temperature_mean');
    return value;
  }
  
  var value_list = ee.List(order_months.map(sample_fn));
  var prop_dct = ee.Dictionary.fromLists(string_months, value_list);
  var prop_dct = prop_dct.set('stationID', id);
  return ee.Feature(null, prop_dct);
}

var out_fc = ee.FeatureCollection(points_fc.map(point_fn));

print(out_fc.first());

Export.table.toDrive({
  collection:out_fc,
  selectors:['stationID', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12'],
  folder:'GEE_Downloads',
  description:'ERA_CN_DEWPT',
});


